In [ ]:
# Google Colab Setup
# Run this cell first to install dependencies
!pip install -q \
    langchain langchain-core langchain-openai langchain-anthropic \
    langchain-google-genai langchain-google-vertexai langchain-community \
    langchain-tavily langchain-text-splitters langchain-model-profiles \
    langgraph langsmith python-dotenv \
    mcp langchain-mcp-adapters \
    pypdf ipywidgets tqdm tavily


In [ ]:
# API Key Setup
# In Google Colab: open the Secrets panel (🔑 icon in the left sidebar),
# add each key by name, and enable notebook access.
# Locally: keys are loaded from your .env file automatically.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    for key in ["ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "TAVILY_API_KEY",
                "LANGSMITH_API_KEY", "LANGSMITH_TRACING", "LANGSMITH_PROJECT"]:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()


In [ ]:
# Download resource file needed for this notebook
import os, urllib.request
os.makedirs('resources', exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/sean-peden/lca-lc-foundations/claude/python-notebooks-ipad-Z3SuX/notebooks/module-3/resources/Chinook.db',
    'resources/Chinook.db'
)
print('Downloaded Chinook.db')


In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)